# Normalizing Flow: 可逆変換で密度を計算する

Normalizing Flowは、単純な分布とデータ分布を可逆変換で結び、変数変換公式により尤度を計算する。


## このノートの読み方

想定読者: 確率密度、正規分布、行列式、MLPの線形変換を理解した学生。

MLPの次に読む教材として、直感、数式、shape、コード、HTMLアニメーション、`Trainer`学習例を往復しながら読む。


## MLPからの橋渡し

普通のMLPは情報を捨てることがある。Flowは可逆性を保つ代わりに、密度評価と生成を両方できる。


## 到達目標

- 変数変換公式を説明できる
- log determinantの意味を説明できる
- NLLをTrainerのlossにできる


## 重要語句

- `invertible`: 逆変換できること
- `log determinant`: 体積変化のlog補正
- `coupling layer`: 半分を固定して残りを変換する可逆層


## 準備

すべてのコードは小さなテンソルで概念を確認するためのものです。長い学習は行いません。


In [ ]:
from __future__ import annotations

import math

import numpy as np
import torch
from jaxtyping import Float
from torch import nn
from torch.utils.data import Dataset
import transformers
from transformers import Trainer, TrainingArguments

SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)

print("PyTorch:", torch.__version__)
print("Transformers:", transformers.__version__)


## shape表

数式を読む前に、どのテンソルがどのshapeを持つかを固定する。

| 記号 | shape | 意味 |
|---|---|---|
| x | (B, 2) | データ |
| z | (B, 2) | base分布上の値 |
| log_det | (B,) | 体積変化補正 |


## レビュー指摘を踏まえた補強

| 観点 | 補足 |
|---|---|
| 向きの固定 | 密度評価ではデータ`x`をbase変数`z=f^{-1}(x)`へ戻す。生成では`z`をサンプルして`x=f(z)`へ進む。 |
| logdetの符号 | コードが逆写像`x -> z`を計算するなら、logdetも逆写像の体積変化として足す。順方向とは符号が逆になることがある。 |
| couplingの制約 | 1層では一部の成分を固定するため表現力が弱い。実用Flowではcoupling層とpermuteを積み重ねる。 |
| 実データ例 | 測定値分布を単純な正規分布へ写す、または単純分布から細胞状態分布を生成する、という見方ができる。 |


## Change of variables

空間が伸び縮みすると密度も変わる。

$$
\log p_X(x)=\log p_Z(f^{-1}(x))+\log |\det J_{f^{-1}}(x)|
$$


## Affine coupling

片方を固定し、もう片方だけをscale/shiftする。

$$
y_1=x_1,\quad y_2=x_2\exp(s(x_1))+t(x_1)
$$


## Maximum likelihood

Flowは負の対数尤度を直接lossにできる。

$$
L(\theta)=-\frac{1}{B}\sum_i\log p_\theta(x_i)
$$


## 小さいテンソルで確認する

次のコードは、上の式がどのshapeを返すかを確認するための最小例である。


In [ ]:
x = torch.tensor([[1.0, 0.5], [0.2, -0.3]])
s = torch.tensor([[0.1], [-0.2]])
t = torch.tensor([[0.3], [0.1]])
x1, x2 = x[:, :1], x[:, 1:]
z1 = x1
z2 = (x2 - t) * torch.exp(-s)
z = torch.cat([z1, z2], dim=1)
log_det = -s.squeeze(1)
base_log_prob = -0.5 * torch.sum(z ** 2 + math.log(2 * math.pi), dim=1)
print("z:", z.round(decimals=3))
print("log_prob:", (base_log_prob + log_det).round(decimals=3))


## 難所HTMLスライド

数式だけでは混ざりやすい箇所を、スライド形式で確認する。各スライドでは入力shape、計算、lossまたは生成手順への接続を1つずつ見る。


<p><a href="../demos/flow_difficulty_slides.html?v=20260522" target="_blank" rel="noopener">別タブで難所スライドを開く</a>（リポジトリ内: <code>demos/flow_difficulty_slides.html</code>）</p>
<iframe
  src="../demos/flow_difficulty_slides.html?v=20260522"
  width="100%"
  height="720"
  style="border: 1px solid #d7dde5; border-radius: 8px;"
  loading="eager"
  title="Normalizing Flow: 可逆変換で密度を計算する difficulty slides"
></iframe>


## HTMLアニメーションで確認する

以下のHTMLは`teaching-html-animation` skillの方針に合わせ、各状態を式・shape・コード上の概念に結びつけている。


### 1D density stretch animation

- 学習目標: 幅が伸びると密度が薄くなる
- 誤解の防止: 点の数だけが密度だと思う

対応する式:

$$
p_X(x)=p_Z(z)|dz/dx|
$$


<p><a href="../demos/flow_density_stretch.html?v=20260522" target="_blank" rel="noopener">別タブで単体HTMLを開く</a>（リポジトリ内: <code>demos/flow_density_stretch.html</code>）</p>
<iframe
  src="../demos/flow_density_stretch.html?v=20260522"
  width="100%"
  height="760"
  style="border: 1px solid #d7dde5; border-radius: 8px;"
  loading="eager"
  title="1D density stretch animation"
></iframe>


### affine coupling animation

- 学習目標: 半分固定して半分変換
- 誤解の防止: 普通のMLPと同じと思う

対応する式:

$$
y_2=x_2\exp(s)+t
$$


<p><a href="../demos/flow_coupling.html?v=20260522" target="_blank" rel="noopener">別タブで単体HTMLを開く</a>（リポジトリ内: <code>demos/flow_coupling.html</code>）</p>
<iframe
  src="../demos/flow_coupling.html?v=20260522"
  width="100%"
  height="760"
  style="border: 1px solid #d7dde5; border-radius: 8px;"
  loading="eager"
  title="affine coupling animation"
></iframe>


### forward inverse toggle animation

- 学習目標: 生成方向と密度評価方向を切り替える
- 誤解の防止: 向きが混ざる

対応する式:

$$
z=f^{-1}(x),\ x=f(z)
$$


<p><a href="../demos/flow_inverse.html?v=20260522" target="_blank" rel="noopener">別タブで単体HTMLを開く</a>（リポジトリ内: <code>demos/flow_inverse.html</code>）</p>
<iframe
  src="../demos/flow_inverse.html?v=20260522"
  width="100%"
  height="760"
  style="border: 1px solid #d7dde5; border-radius: 8px;"
  loading="eager"
  title="forward inverse toggle animation"
></iframe>


### logdet area animation

- 学習目標: 面積変化とlogdetを対応させる
- 誤解の防止: detがなぜ必要か分からない

対応する式:

$$
\log|\det J|
$$


<p><a href="../demos/flow_logdet.html?v=20260522" target="_blank" rel="noopener">別タブで単体HTMLを開く</a>（リポジトリ内: <code>demos/flow_logdet.html</code>）</p>
<iframe
  src="../demos/flow_logdet.html?v=20260522"
  width="100%"
  height="760"
  style="border: 1px solid #d7dde5; border-radius: 8px;"
  loading="eager"
  title="logdet area animation"
></iframe>


## `Trainer`で学習する

この章の`Trainer`例は、汎用MSE回帰ではなく、`Normalizing Flow: 可逆変換で密度を計算する`固有のデータ形式とlossを返す。基本は標準の`Trainer(model, args, train_dataset)`を使い、`forward`が`loss`と`logits`を返す形にそろえる。GANはD/Gでoptimizerを分ける必要があるため`Trainer`を継承した交互更新デモ、DBMは平均場CDサロゲートとして扱う。


In [ ]:
class TinyFlowDataset(Dataset):
    def __init__(self, n_samples: int = 32) -> None:
        self.x = torch.randn(n_samples, 2) * torch.tensor([0.4, 1.2]) + torch.tensor([1.0, -0.5])

    def __len__(self) -> int:
        return len(self.x)

    def __getitem__(self, index: int) -> dict[str, torch.Tensor]:
        return {"x": self.x[index]}


class TrainerAffineFlow(nn.Module):
    def __init__(self) -> None:
        super().__init__()
        self.s_net = nn.Linear(1, 1)
        self.t_net = nn.Linear(1, 1)

    def forward(self, x: torch.Tensor) -> dict[str, torch.Tensor]:
        x1, x2 = x[:, :1], x[:, 1:]
        s = torch.tanh(self.s_net(x1))
        t = self.t_net(x1)
        z1 = x1
        z2 = (x2 - t) * torch.exp(-s)
        z = torch.cat([z1, z2], dim=1)
        log_det = -torch.sum(s, dim=1)
        base_log_prob = -0.5 * torch.sum(z ** 2 + math.log(2 * math.pi), dim=1)
        log_prob = base_log_prob + log_det
        loss = -torch.mean(log_prob)
        return {"loss": loss, "logits": z}


training_args = TrainingArguments(
    output_dir="./results/flow_trainer_demo",
    max_steps=1,
    per_device_train_batch_size=8,
    learning_rate=1e-3,
    logging_strategy="no",
    save_strategy="no",
    report_to="none",
    disable_tqdm=True,
    seed=SEED,
    use_cpu=not torch.cuda.is_available(),
)

trainer = Trainer(model=TrainerAffineFlow(), args=training_args, train_dataset=TinyFlowDataset())
train_output = trainer.train()
print("Flow NLL Trainer loss:", train_output.training_loss)


with torch.no_grad():
    z = torch.randn(3, 2)
    z1, z2 = z[:, :1], z[:, 1:]
    s = torch.tanh(trainer.model.s_net(z1))
    t = trainer.model.t_net(z1)
    x_sample = torch.cat([z1, z2 * torch.exp(s) + t], dim=1)
print("flow sample shape:", x_sample.shape)


## 生成モデル間の比較

| モデル | 学習目的 | 尤度 | 生成手順 | 代表的な弱点 |
|---|---|---|---|---|
| VAE/IWAE | ELBO / IWAE bound | 下界 | Decoderにzを入れる | ぼやけ、推論分布の設計 |
| GAN | Dをだます | 通常は不可 | G(z)を一発生成 | mode collapse、不安定 |
| Flow | NLL | 厳密 | 可逆変換の順方向 | 可逆層の制約 |
| RBM/DBM | エネルギー差 | 分配関数が困難 | Gibbs sampling | 近似推論が重い |
| Diffusion/DDPM | score/noise予測 | 目的により異なる | 多段denoising | samplingが遅い |


## 発展課題

- forward/inverseを両方実装する
- coupling層を2層にする
- RealNVPやGlowへ進む


## 確認問題

- log determinantは何を補正するか。
- Flowで普通のMLPをそのまま使いにくい理由を書く。


## まとめ

- MLPから何が変わったのかを、shapeとlossで確認する。
- HTMLアニメーションは式の代わりではなく、式とコードを読むための補助である。
- `Trainer`は学習ループを隠すが、`Dataset`のキー、`forward`の引数、`loss`の意味は必ず確認する。
